# 07 · Route tool-calling turns across providers

*Does routing the agent's tool-calling turns beat pinning one model, now that the pool spans OpenAI, OSS, and Anthropic?*

Agentic routing selects a model **per turn** in a tool-calling loop, so planning
turns and synthesis turns can land on different models. With the pool spanning
providers, we test whether per-turn routing beats pinning a single frontier model on
a multi-step travel task.

> **Skeleton.** This rung is scaffolded: the header, scoring toolkit, and scorecard
> are ready, and the lever cell has a working starting point to refine as you test.

```mermaid
flowchart LR
    Pin[Pinned frontier agent] --> S[Tool-calling benchmark]
    Route[Agentic router agent] --> S
    S --> Sc[Quality · cost · turns · model mix]
    Sc --> D{Per-turn routing wins<br/>on cost at equal quality?}
```

## 1. Install dependencies and load config

In [ ]:
## 1. Ensure dependencies and read .env
import importlib.util, subprocess, sys

_needed = {
    "azure-ai-projects": "azure.ai.projects", "azure-identity": "azure.identity",
    "openai": "openai", "python-dotenv": "dotenv", "pandas": "pandas", "matplotlib": "matplotlib",
}
_missing = [pkg for pkg, mod in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *_missing])
else:
    print("All dependencies present.")

from dotenv import load_dotenv
load_dotenv(".env")
print(".env loaded from this folder (if present).")

## 2. Authenticate and open the project

In [ ]:
## 2. Project client
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = os.environ["MICROSOFT_FOUNDRY_ENDPOINT"]
credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=endpoint, credential=credential)
print("Project client ready.")

## 3. Scoring toolkit

Reuses the shared [Contoso benchmark](../../../demos/contoso-travel/benchmark/) so the
workload stays fixed across every rung.

In [ ]:
## 3. Scoring toolkit (shared shape with notebooks 01–02)
import sys, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../../../demos/contoso-travel/benchmark").resolve()))
from run_benchmark import load_json, grade  # noqa: E402

BENCH = Path("../../../demos/contoso-travel/benchmark")
QUERIES = load_json(BENCH / "queries.json")["queries"]

def run_suite(ask, label, queries=QUERIES, limit=None) -> pd.DataFrame:
    rows = []
    for q in (queries[:limit] if limit else queries):
        turns = q.get("turns") or [q["query"]]
        start = time.perf_counter()
        try:
            text, model, tin, tout = ask(turns)
        except Exception as exc:
            text, model, tin, tout = f"ERROR: {exc}", "", 0, 0
        passed, _ = grade(q, text)
        rows.append({"run": label, "id": q["id"], "category": q["category"], "gate": q["gate"],
                     "model": model or "-", "passed": passed,
                     "latency_ms": round((time.perf_counter() - start) * 1000),
                     "tokens_in": tin, "tokens_out": tout})
    return pd.DataFrame(rows)

def make_model_caller(deployment: str):
    """Call a fixed deployment (or a named router deployment) directly."""
    client = project.get_openai_client()
    def ask(turns):
        conv = client.conversations.create().id
        text = model = ""; tin = tout = 0
        for turn in turns:
            resp = client.responses.create(conversation=conv, input=turn, model=deployment)
            text = resp.output_text
            model = getattr(resp, "model", "") or deployment
            u = getattr(resp, "usage", None)
            if u:
                tin += getattr(u, "input_tokens", 0) or 0
                tout += getattr(u, "output_tokens", 0) or 0
        return text, model, tin, tout
    return ask

def compare(frames, order=None):
    allruns = pd.concat(frames, ignore_index=True)
    summary = allruns.groupby("run").agg(
        quality_pct=("passed", lambda s: round(100 * s.mean(), 1)),
        policy_pct=("gate", lambda s: None),
        avg_tokens=("tokens_out", "mean"),
        p50_ms=("latency_ms", "median"),
    )
    # policy accuracy computed separately (gate == 'policy')
    pol = allruns[allruns["gate"] == "policy"].groupby("run")["passed"].mean().mul(100).round(1)
    summary["policy_pct"] = pol
    if order:
        summary = summary.reindex([o for o in order if o in summary.index])
    return allruns, summary

## 4. Change the lever

Build an agent with tools (mock the booking tools to avoid side effects) on the
`model-router` deployment, and compare against the same agent pinned to the frontier.
Capture the model chosen per turn to show cross-provider decomposition.

In [ ]:
## 4. Define pinned vs agentic-routing agents
# Reuse the tool-enabled agent from your project; here we compare its model binding.
configs = {"pinned-frontier": os.getenv("AZURE_FRONTIER_DEPLOYMENT"),
           "agentic-router": os.getenv("AZURE_MODEL_ROUTER_DEPLOYMENT")}
configs = {k: v for k, v in configs.items() if v}
print("Configurations:", configs)
# TODO: attach mock booking tools and record response.model per tool-calling turn.

## 5. Score and compare

Run each configuration over the benchmark, then read the four-dimension scorecard —
quality, policy accuracy, cost (tokens), latency — plus the selected-model mix.

In [ ]:
## 5. Run the comparison
# TODO: build `frames` by running run_suite(...) for each configuration above.
# Example once the lever cell defines the callers/deployments:
#   frames = [run_suite(make_model_caller(dep), label) for label, dep in configs.items()]
#   allruns, summary = compare(frames, order=list(configs))
#   display(summary)
frames = []
if frames:
    allruns, summary = compare(frames)
    display(summary)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    summary["quality_pct"].plot.bar(ax=ax[0], color="#4C78A8", title="Quality %"); ax[0].set_ylim(0, 100)
    summary["avg_tokens"].plot.bar(ax=ax[1], color="#E45756", title="Avg output tokens (cost)")
    plt.tight_layout(); plt.show()
else:
    print("Fill in the lever cell and the frames list above, then re-run.")

## 6. Your Turn

- **Inspect the loop.** Which turns went to which provider? Do cheap models handle
  tool plumbing while a strong model does synthesis?
- **Check side effects.** Are your booking tools mocked so optimization runs are safe?

## 7. Summary

Per-turn routing decomposed the tool-calling loop across providers, cutting cost at
matched quality. Finally, **notebook 08** refreshes the model pool and gives each
agent its own routing policy.

## 8. References

- [Use model router with Foundry agents](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/model-router-agents)
- [Model router release history](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/whats-new-model-router)
- [Capsule overview](README.md) · [Glossary](../../../docs/GLOSSARY.md)